# 01 数据准备与清洗

**项目：** 电商用户行为分析与购买预测
**数据：** 阿里天池"淘宝用户行为数据集"（User Behavior Data from Taobao for Recommendation），2017 年 11 月 25 日至 12 月 3 日，约一百万名随机用户、约一亿条行为记录。

| 字段 | 含义 |
|---|---|
| user_id | 用户编号 |
| item_id | 商品编号 |
| category_id | 商品类目编号 |
| behavior_type | 行为类型：pv 浏览、fav 收藏、cart 加购、buy 购买 |
| ts | 行为发生的时间戳（秒） |

**这一步要做的事：**
1. 用 DuckDB 直接读取 3GB 以上的 CSV，检查数据质量（缺失、重复、时间越界、异常行为类型）
2. 清洗数据，把时间戳转成北京时间，补充日期、小时、是否周末等字段
3. 保存为 Parquet 格式，后面几个笔记本直接读取，速度快很多

In [1]:
# 在 Google Colab 上运行时先执行这一格；在自己电脑上运行会自动跳过
import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/ecommerce-user-behavior/notebooks")
    subprocess.run("pip -q install duckdb xgboost", shell=True, check=True)
    subprocess.run("apt-get -qq install -y fonts-noto-cjk > /dev/null", shell=True, check=True)
    import matplotlib.font_manager as fm
    for f in fm.findSystemFonts():
        if "NotoSansCJK" in f:
            fm.fontManager.addfont(f)
    print("Colab 环境准备完成，当前目录：", os.getcwd())

Mounted at /content/drive
Colab 环境准备完成，当前目录： /content/drive/MyDrive/ecommerce-user-behavior/notebooks


In [2]:
import sys
sys.path.append("..")

import time
import pandas as pd
import config, utils

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# 想先小规模试跑，去掉下一行开头的 # 号（按用户抽样 10%）。只有这个笔记本用到抽样比例。
config.SAMPLE_FRACTION = 0.1

con = utils.connect()

assert config.RAW_CSV.exists(), (
    f"没找到原始数据：{config.RAW_CSV}\n"
    "请先从天池下载 UserBehavior.csv 放到 data/raw/ 目录；"
    "只想测试代码的话，在项目根目录运行 python make_sample_data.py"
)
print(f"原始文件大小：{config.RAW_CSV.stat().st_size / 1024**3:.2f} GB")
print(f"抽样比例：{config.SAMPLE_FRACTION}")

原始文件大小：3.42 GB
抽样比例：0.1


## 1. 读取原始数据并检查质量

原始 CSV 没有表头，读取时手动指定字段名和类型。DuckDB 不会把整个文件读进内存，所以一亿行也能在普通笔记本电脑上处理。

In [3]:
t0 = time.time()
con.execute(f'''
    CREATE OR REPLACE VIEW raw AS
    SELECT *
    FROM read_csv('{config.RAW_CSV.as_posix()}',
                  header = false,
                  columns = {{'user_id': 'BIGINT', 'item_id': 'BIGINT', 'category_id': 'BIGINT',
                             'behavior_type': 'VARCHAR', 'ts': 'BIGINT'}})
''')

quality = con.execute(f'''
    SELECT
        COUNT(*)                                        AS 总行数,
        COUNT(DISTINCT user_id)                         AS 用户数,
        COUNT(DISTINCT item_id)                         AS 商品数,
        COUNT(DISTINCT category_id)                     AS 类目数,
        COUNT(*) FILTER (WHERE user_id IS NULL OR item_id IS NULL OR category_id IS NULL
                            OR behavior_type IS NULL OR ts IS NULL)                AS 含缺失值行数,
        COUNT(*) FILTER (WHERE behavior_type NOT IN ('pv', 'fav', 'cart', 'buy'))  AS 异常行为类型行数,
        COUNT(*) FILTER (WHERE epoch_ms((ts + 8 * 3600) * 1000) <  TIMESTAMP '{config.START_DATE}'
                            OR epoch_ms((ts + 8 * 3600) * 1000) >= TIMESTAMP '{config.REFERENCE_DATE}')
                                                        AS 时间越界行数,
        MIN(epoch_ms((ts + 8 * 3600) * 1000))           AS 最早时间,
        MAX(epoch_ms((ts + 8 * 3600) * 1000))           AS 最晚时间
    FROM raw
''').df()

dup_rows = con.execute('''
    SELECT COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM raw)) FROM raw
''').fetchone()[0]
quality["完全重复行数"] = dup_rows
print(f"耗时 {time.time() - t0:.1f} 秒")
quality.T.rename(columns={0: "值"})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

耗时 274.6 秒


,值
总行数,100150807
用户数,987994
商品数,4162024
类目数,9439
含缺失值行数,0
异常行为类型行数,0
时间越界行数,55576
最早时间,1902-05-08 06:32:46
最晚时间,2037-04-09 13:22:35
完全重复行数,49


In [4]:
behavior_dist = con.execute('''
    SELECT behavior_type AS 行为类型, COUNT(*) AS 行数,
           COUNT(*) / SUM(COUNT(*)) OVER () AS 占比
    FROM raw
    GROUP BY behavior_type
    ORDER BY 行数 DESC
''').df()
behavior_dist

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,行为类型,行数,占比
0,pv,89716264,0.8958
1,cart,5530446,0.0552
2,fav,2888258,0.0288
3,buy,2015839,0.0201


## 2. 清洗规则

| 问题 | 处理方式 | 原因 |
|---|---|---|
| 时间戳是 UTC 秒数 | 加 8 小时转成北京时间 | 用户行为的小时规律要按当地时间看 |
| 时间越界 | 只保留 11 月 25 日至 12 月 3 日 | 数据集说明里的有效范围，越界记录多为采集错误 |
| 完全重复的记录 | 去重 | 同一用户、同一商品、同一行为、同一秒，视为重复上报 |
| 缺失值、异常行为类型 | 删除 | 无法使用 |
| 数据量过大 | 可选：按用户哈希抽样 | 按用户抽而不是按行抽，才能保留每个用户完整的行为序列 |

In [5]:
t0 = time.time()
config.CLEAN_PARQUET.parent.mkdir(parents=True, exist_ok=True)
sample_cond = (
    "TRUE" if config.SAMPLE_FRACTION >= 1
    else f"hash(user_id) % 1000 < {int(config.SAMPLE_FRACTION * 1000)}"
)

con.execute(f'''
    COPY (
        WITH dedup AS (
            SELECT DISTINCT user_id, item_id, category_id, behavior_type, ts
            FROM raw
            WHERE user_id IS NOT NULL AND item_id IS NOT NULL AND category_id IS NOT NULL
              AND ts IS NOT NULL
              AND behavior_type IN ('pv', 'fav', 'cart', 'buy')
              AND {sample_cond}
        ),
        timed AS (
            SELECT *, epoch_ms((ts + 8 * 3600) * 1000) AS event_time
            FROM dedup
        )
        SELECT
            user_id, item_id, category_id, behavior_type, ts, event_time,
            CAST(event_time AS DATE)              AS event_date,
            hour(event_time)                      AS event_hour,
            isodow(event_time)                    AS weekday,      -- 1 = 周一，7 = 周日
            isodow(event_time) IN (6, 7)          AS is_weekend
        FROM timed
        WHERE event_time >= TIMESTAMP '{config.START_DATE}'
          AND event_time <  TIMESTAMP '{config.REFERENCE_DATE}'
        ORDER BY user_id, ts
    ) TO '{config.CLEAN_PARQUET.as_posix()}' (FORMAT parquet)
''')
print(f"清洗完成，耗时 {time.time() - t0:.1f} 秒 -> {config.CLEAN_PARQUET}")

sampled_rows = None if config.SAMPLE_FRACTION >= 1 else int(
    con.execute(f"SELECT COUNT(*) FROM raw WHERE {sample_cond}").fetchone()[0])

con = utils.connect()   # 重新连接，注册清洗后的视图 ub

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

清洗完成，耗时 78.1 秒 -> /content/drive/MyDrive/ecommerce-user-behavior/data/processed/user_behavior_clean.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
clean = con.execute('''
    SELECT COUNT(*) AS 行数, COUNT(DISTINCT user_id) AS 用户数,
           COUNT(DISTINCT item_id) AS 商品数, COUNT(DISTINCT category_id) AS 类目数,
           MIN(event_time) AS 最早时间, MAX(event_time) AS 最晚时间
    FROM ub
''').df()

raw_rows = int(quality.loc[0, "总行数"])
# 抽样时，"清洗前"指抽中用户的原始行数，这样删除比例只反映清洗本身
sampled_rows = sampled_rows or raw_rows
clean_rows = int(clean.loc[0, "行数"])
if sampled_rows < raw_rows:
    print(f"按用户抽样 {config.SAMPLE_FRACTION:.0%}：原始 {raw_rows:,} 行，抽中用户共 {sampled_rows:,} 行")
print(f"清洗前 {sampled_rows:,} 行，清洗后 {clean_rows:,} 行，"
      f"删除 {sampled_rows - clean_rows:,} 行（{utils.pct(1 - clean_rows / sampled_rows)}）")
clean.T.rename(columns={0: "值"})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

按用户抽样 10%：原始 100,150,807 行，抽中用户共 10,016,115 行
清洗前 10,016,115 行，清洗后 10,010,718 行，删除 5,397 行（0.05%）


,值
行数,10010718
用户数,99057
商品数,1594042
类目数,8006
最早时间,2017-11-25 00:00:00
最晚时间,2017-12-03 23:59:59


In [7]:
daily_rows = con.execute('''
    SELECT event_date AS 日期, COUNT(*) AS 行数, COUNT(DISTINCT user_id) AS 活跃用户数
    FROM ub GROUP BY event_date ORDER BY event_date
''').df()
daily_rows

,日期,行数,活跃用户数
0,2017-11-25,1046947,70753
1,2017-11-26,1061547,71659
2,2017-11-27,1009050,71009
3,2017-11-28,985142,71174
4,2017-11-29,1032489,72118
5,2017-11-30,1044300,73066
6,2017-12-01,1090500,74036
7,2017-12-02,1372208,97264
8,2017-12-03,1368535,96973


In [8]:
utils.update_summary("01_数据清洗", {
    "抽样比例": config.SAMPLE_FRACTION,
    "原始行数": raw_rows,
    "抽样后行数": sampled_rows,
    "清洗后行数": clean_rows,
    "清洗删除行数": sampled_rows - clean_rows,
    "时间越界行数": int(quality.loc[0, "时间越界行数"]),
    "完全重复行数": int(dup_rows),
    "用户数": int(clean.loc[0, "用户数"]),
    "商品数": int(clean.loc[0, "商品数"]),
    "类目数": int(clean.loc[0, "类目数"]),
})
print("已写入 results/summary.json")

已写入 results/summary.json


## 小结

- 记录清洗前后的行数和删除原因，面试时常被问"你怎么做数据清洗"，这里的表格就是答案。
- 下一步：`02_SQL_漏斗与留存分析.ipynb`